# 子波振幅增益曲线（深度域输出）

基于 `wavelet_batch_synthetic_depth@20260427.ipynb` 的批量合成记录 QC 输出，先在时间域计算每口井的平滑振幅缩放曲线：

```text
seismic_norm(t) ~= gain(t) * synthetic_raw(t)
```

再通过每口井的 `depth_shift_curve_*.csv` 将 `gain(t)` 映射到 TVDSS 深度域，输出每井 `gain_curve_*.csv`、汇总表和 QC 图。空间建模（普通克里金、协同克里金或地震属性约束建模）放到后续独立流程中完成。


In [ ]:
import traceback
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if not (repo_root / "src").exists():
    raise RuntimeError("Could not locate repo root containing 'src'.")

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 100)


## 1) 配置输入输出


In [ ]:
data_root = repo_root / "data"

batch_output_dir = data_root / "output_wavelet_batch_synthetic_depth_20260427"
batch_metrics_file = batch_output_dir / "wavelet_batch_metrics.csv"

output_dir = data_root / "output_wavelet_amplitude_gain_curve_depth_20260428"
gain_curve_dir = output_dir / "gain_curves"
figure_dir = output_dir / "figures"
gain_metrics_file = output_dir / "wavelet_amplitude_gain_curve_metrics.csv"

# Gain estimation parameters.
gain_window_s = 0.200
min_valid_fraction = 0.25
eps_fraction = 1e-3
synthetic_energy_percentile = 95.0
gain_floor = 1e-6

# Depth sampling for exported gain(z) curves.
depth_sample_step_m = 5.0

required_paths = [batch_metrics_file]
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(path)

output_dir.mkdir(parents=True, exist_ok=True)
gain_curve_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

batch_metrics_df = pd.read_csv(batch_metrics_file)
ok_metrics_df = batch_metrics_df.loc[batch_metrics_df["status"] == "ok"].copy()
if ok_metrics_df.empty:
    raise ValueError("No successful wells found in wavelet batch metrics.")

print("Batch metrics:", batch_metrics_file)
print("Successful wells:", len(ok_metrics_df))
print("Output dir:", output_dir)


## 2) 工具函数


In [ ]:
def safe_name(well_name: str) -> str:
    return well_name.replace("/", "_").replace("\\", "_").replace(" ", "_")


def resolve_artifact_path(path_value: str | Path) -> Path:
    path = Path(path_value)
    if path.is_absolute():
        return path
    return repo_root / path


def centered_moving_sum(values: np.ndarray, window_samples: int) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    window_samples = int(max(window_samples, 1))
    if values.size == 0:
        return values.copy()
    radius_left = window_samples // 2
    radius_right = window_samples - 1 - radius_left
    padded = np.pad(values, (radius_left, radius_right), mode="constant", constant_values=0.0)
    cumsum = np.concatenate([[0.0], np.cumsum(padded)])
    return cumsum[window_samples:] - cumsum[:-window_samples]


def resolve_window_samples(twt_s: np.ndarray, window_s: float) -> int:
    dt_s = float(np.nanmedian(np.diff(np.asarray(twt_s, dtype=float))))
    if not np.isfinite(dt_s) or dt_s <= 0.0:
        raise ValueError("Cannot resolve gain window because TWT sampling is invalid.")
    window_samples = int(round(float(window_s) / dt_s))
    window_samples = max(window_samples, 3)
    if window_samples % 2 == 0:
        window_samples += 1
    return window_samples


def compute_trace_metrics(reference: np.ndarray, prediction: np.ndarray, mask: np.ndarray) -> dict:
    reference = np.asarray(reference, dtype=float)
    prediction = np.asarray(prediction, dtype=float)
    mask = np.asarray(mask, dtype=bool) & np.isfinite(reference) & np.isfinite(prediction)
    n = int(mask.sum())
    if n < 5:
        return {"corr": np.nan, "nrmse": np.nan, "n_samples": n}

    ref = reference[mask]
    pred = prediction[mask]
    if np.std(ref) <= 0.0 or np.std(pred) <= 0.0:
        corr = np.nan
    else:
        corr = float(np.corrcoef(ref, pred)[0, 1])

    denom = float(np.linalg.norm(ref))
    nrmse = np.nan if denom <= 0.0 else float(np.linalg.norm(ref - pred) / denom)
    return {"corr": corr, "nrmse": nrmse, "n_samples": n}


def interpolate_positive_gain(twt_s: np.ndarray, gain_raw: np.ndarray, valid_mask: np.ndarray) -> np.ndarray:
    twt_s = np.asarray(twt_s, dtype=float)
    gain_raw = np.asarray(gain_raw, dtype=float)
    valid_mask = np.asarray(valid_mask, dtype=bool) & np.isfinite(twt_s) & np.isfinite(gain_raw) & (gain_raw > 0.0)
    if int(valid_mask.sum()) < 2:
        raise ValueError("Need at least two positive valid gain samples for interpolation.")
    return np.interp(twt_s, twt_s[valid_mask], gain_raw[valid_mask])


def smooth_gain_in_log_domain(gain_values: np.ndarray, window_samples: int) -> np.ndarray:
    gain_values = np.asarray(gain_values, dtype=float)
    safe_gain = np.maximum(gain_values, gain_floor)
    log_gain = np.log(safe_gain)
    numerator = centered_moving_sum(log_gain, window_samples)
    denominator = centered_moving_sum(np.ones_like(log_gain), window_samples)
    return np.exp(numerator / denominator)


def map_gain_to_regular_depth(
    twt_s: np.ndarray, gain_smooth: np.ndarray, depth_shift_df: pd.DataFrame, dz_m: float
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    required_columns = {"twt_s", "tvdss_m"}
    missing = required_columns - set(depth_shift_df.columns)
    if missing:
        raise ValueError(f"Depth shift curve missing columns: {sorted(missing)}")

    depth_twt = depth_shift_df["twt_s"].to_numpy(dtype=float)
    depth_z = depth_shift_df["tvdss_m"].to_numpy(dtype=float)
    finite_depth = np.isfinite(depth_twt) & np.isfinite(depth_z)
    depth_twt = depth_twt[finite_depth]
    depth_z = depth_z[finite_depth]
    if depth_twt.size < 2:
        raise ValueError("Depth shift curve has too few finite TWT/TVDSS samples.")

    order = np.argsort(depth_twt)
    depth_twt = depth_twt[order]
    depth_z = depth_z[order]
    unique_twt, unique_idx = np.unique(depth_twt, return_index=True)
    unique_z = depth_z[unique_idx]

    tvdss_at_twt = np.interp(twt_s, unique_twt, unique_z, left=unique_z[0], right=unique_z[-1])
    finite_gain = np.isfinite(tvdss_at_twt) & np.isfinite(gain_smooth) & (gain_smooth > 0.0)
    if int(finite_gain.sum()) < 2:
        raise ValueError("Gain curve has too few finite depth-mapped samples.")

    z = tvdss_at_twt[finite_gain]
    g = gain_smooth[finite_gain]
    order = np.argsort(z)
    z = z[order]
    g = g[order]
    unique_z, unique_z_idx = np.unique(z, return_index=True)
    unique_g = g[unique_z_idx]
    if unique_z.size < 2:
        raise ValueError("Depth-mapped gain curve has too few unique TVDSS samples.")

    regular_z = np.arange(float(unique_z[0]), float(unique_z[-1]) + 0.5 * dz_m, float(dz_m))
    regular_gain = np.interp(regular_z, unique_z, unique_g)
    return tvdss_at_twt, regular_z, regular_gain


## 4) 单井振幅增益曲线


In [ ]:
def build_gain_curve(row: pd.Series) -> dict:
    well_name = str(row["well_name"])
    name = safe_name(well_name)
    scale = float(row["scale"])
    if not np.isfinite(scale) or scale <= 0.0:
        raise ValueError(f"well={well_name} has invalid scale={scale}")

    synthetic_qc_path = resolve_artifact_path(row["synthetic_qc_path"])
    depth_shift_curve_path = resolve_artifact_path(row["depth_shift_curve_path"])
    for path in [synthetic_qc_path, depth_shift_curve_path]:
        if not path.exists():
            raise FileNotFoundError(path)

    qc_df = pd.read_csv(synthetic_qc_path)
    depth_shift_df = pd.read_csv(depth_shift_curve_path)
    required_qc_columns = {"twt_s", "seismic_norm", "synthetic_scaled", "eval_mask"}
    missing = required_qc_columns - set(qc_df.columns)
    if missing:
        raise ValueError(f"Synthetic QC {synthetic_qc_path} missing columns: {sorted(missing)}")

    twt_s = qc_df["twt_s"].to_numpy(dtype=float)
    seismic_norm = qc_df["seismic_norm"].to_numpy(dtype=float)
    synthetic_scaled = qc_df["synthetic_scaled"].to_numpy(dtype=float)
    synthetic_raw = synthetic_scaled / scale
    eval_mask = qc_df["eval_mask"].astype(bool).to_numpy()

    finite = np.isfinite(twt_s) & np.isfinite(seismic_norm) & np.isfinite(synthetic_raw)
    base_valid = eval_mask & finite
    if int(base_valid.sum()) < 5:
        raise ValueError(f"well={well_name} has too few valid samples for gain estimation.")

    window_samples = resolve_window_samples(twt_s, gain_window_s)
    synthetic_energy = synthetic_raw**2
    eps = eps_fraction * float(np.nanpercentile(synthetic_energy[base_valid], synthetic_energy_percentile))
    if not np.isfinite(eps) or eps <= 0.0:
        eps = eps_fraction * float(np.nanmean(synthetic_energy[base_valid]))
    if not np.isfinite(eps) or eps <= 0.0:
        raise ValueError(f"well={well_name} cannot resolve a positive gain denominator epsilon.")

    weights = base_valid.astype(float)
    valid_count = centered_moving_sum(weights, window_samples)
    numerator = centered_moving_sum(np.where(base_valid, seismic_norm * synthetic_raw, 0.0), window_samples)
    denominator = centered_moving_sum(np.where(base_valid, synthetic_energy, 0.0), window_samples)
    min_valid_count = max(3, int(np.ceil(window_samples * min_valid_fraction)))

    gain_raw = numerator / (denominator + eps * np.maximum(valid_count, 1.0))
    gain_valid_mask = (
        (valid_count >= min_valid_count)
        & np.isfinite(gain_raw)
        & (gain_raw > 0.0)
        & (denominator > eps * min_valid_count)
    )
    gain_filled = interpolate_positive_gain(twt_s, gain_raw, gain_valid_mask)
    gain_smooth = smooth_gain_in_log_domain(gain_filled, window_samples)
    relative_gain_smooth = gain_smooth / scale
    synthetic_dynamic_gain = gain_smooth * synthetic_raw

    tvdss_m, regular_z, regular_gain = map_gain_to_regular_depth(
        twt_s,
        gain_smooth,
        depth_shift_df,
        depth_sample_step_m,
    )

    gain_curve_df = pd.DataFrame(
        {
            "twt_s": twt_s,
            "tvdss_m": tvdss_m,
            "seismic_norm": seismic_norm,
            "synthetic_raw": synthetic_raw,
            "synthetic_scaled": synthetic_scaled,
            "gain_raw": gain_raw,
            "gain_smooth": gain_smooth,
            "relative_gain_smooth": relative_gain_smooth,
            "eval_mask": eval_mask,
            "gain_valid_mask": gain_valid_mask,
        }
    )
    gain_curve_path = gain_curve_dir / f"gain_curve_{name}.csv"
    gain_curve_df.to_csv(gain_curve_path, index=False)

    gain_depth_df = pd.DataFrame(
        {
            "tvdss_m": regular_z,
            "gain_smooth": regular_gain,
            "relative_gain_smooth": regular_gain / scale,
        }
    )
    gain_depth_path = gain_curve_dir / f"gain_depth_curve_{name}.csv"
    gain_depth_df.to_csv(gain_depth_path, index=False)

    constant_metrics = compute_trace_metrics(seismic_norm, synthetic_scaled, eval_mask)
    dynamic_metrics = compute_trace_metrics(seismic_norm, synthetic_dynamic_gain, eval_mask)
    gain_eval = gain_smooth[eval_mask & np.isfinite(gain_smooth)]
    relative_eval = relative_gain_smooth[eval_mask & np.isfinite(relative_gain_smooth)]
    if gain_eval.size == 0:
        raise ValueError(f"well={well_name} has no finite smoothed gain in eval window.")

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
    t_ms = twt_s * 1000.0
    axes[0].plot(gain_raw, t_ms, lw=0.7, color="0.65", label="raw local LS")
    axes[0].plot(gain_smooth, t_ms, lw=1.3, color="tab:blue", label="smooth gain")
    axes[0].axvline(scale, color="tab:red", lw=1.0, ls="--", label="constant scale")
    axes[0].invert_yaxis()
    axes[0].set_xlabel("Absolute gain")
    axes[0].set_ylabel("Relative TWT (ms)")
    axes[0].set_title(f"{well_name}: gain(t)")
    axes[0].legend(loc="best", fontsize=8)
    axes[0].grid(True, alpha=0.25)

    axes[1].plot(gain_smooth, tvdss_m, lw=1.2, color="tab:green", label="time samples")
    axes[1].plot(regular_gain, regular_z, lw=0.9, color="black", alpha=0.7, label="5 m export")
    axes[1].axvline(scale, color="tab:red", lw=1.0, ls="--")
    axes[1].invert_yaxis()
    axes[1].set_xlabel("Absolute gain")
    axes[1].set_ylabel("TVDSS (m)")
    axes[1].set_title("gain(z)")
    axes[1].legend(loc="best", fontsize=8)
    axes[1].grid(True, alpha=0.25)

    axes[2].plot(seismic_norm, t_ms, lw=0.9, color="black", label="seismic")
    axes[2].plot(synthetic_scaled, t_ms, lw=0.9, color="tab:red", alpha=0.75, label="constant scale")
    axes[2].plot(synthetic_dynamic_gain, t_ms, lw=0.9, color="tab:blue", alpha=0.85, label="dynamic gain")
    axes[2].invert_yaxis()
    axes[2].set_xlabel("Normalized amplitude")
    axes[2].set_title("synthetic comparison")
    axes[2].legend(loc="best", fontsize=8)
    axes[2].grid(True, alpha=0.25)

    fig.tight_layout()
    gain_fig_path = figure_dir / f"qc_{name}_gain_curve.png"
    fig.savefig(gain_fig_path, dpi=180, bbox_inches="tight")
    plt.close(fig)

    return {
        "well_name": well_name,
        "status": "ok",
        "error": "",
        "inline": float(row["inline_float"]),
        "xline": float(row["xline_float"]),
        "scale": scale,
        "signed_ls_scale": float(row["signed_ls_scale"]),
        "gain_median": float(np.nanmedian(gain_eval)),
        "gain_mean": float(np.nanmean(gain_eval)),
        "gain_p10": float(np.nanpercentile(gain_eval, 10)),
        "gain_p90": float(np.nanpercentile(gain_eval, 90)),
        "relative_gain_median": float(np.nanmedian(relative_eval)) if relative_eval.size else np.nan,
        "constant_corr": constant_metrics["corr"],
        "constant_nrmse": constant_metrics["nrmse"],
        "dynamic_gain_corr": dynamic_metrics["corr"],
        "dynamic_gain_nrmse": dynamic_metrics["nrmse"],
        "n_eval_samples": int(constant_metrics["n_samples"]),
        "n_gain_valid_samples": int(gain_valid_mask.sum()),
        "gain_window_s": float(gain_window_s),
        "window_samples": int(window_samples),
        "eps": float(eps),
        "gain_depth_min_m": float(regular_z[0]),
        "gain_depth_max_m": float(regular_z[-1]),
        "gain_curve_path": str(gain_curve_path),
        "gain_depth_curve_path": str(gain_depth_path),
        "gain_fig_path": str(gain_fig_path),
    }


gain_metric_rows = []
for _, row in ok_metrics_df.iterrows():
    well_name = row["well_name"]
    print(f"\n=== {well_name} ===")
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            metric_row = build_gain_curve(row)
        print(
            f"OK: gain_median={metric_row['gain_median']:.3f}, "
            f"relative={metric_row['relative_gain_median']:.3f}, "
            f"nrmse {metric_row['constant_nrmse']:.3f}->{metric_row['dynamic_gain_nrmse']:.3f}"
        )
    except Exception as exc:
        metric_row = {"well_name": well_name, "status": "failed", "error": str(exc)}
        print("FAILED:", exc)
        traceback.print_exc(limit=2)
    gain_metric_rows.append(metric_row)

gain_metrics_df = pd.DataFrame(gain_metric_rows)
gain_metrics_df.to_csv(gain_metrics_file, index=False)
print("\nSaved", gain_metrics_file)
display(gain_metrics_df)

if not (gain_metrics_df["status"] == "ok").any():
    raise ValueError("No successful gain curves were built.")


## 5) 汇总 QC


In [ ]:
ok_gain_df = gain_metrics_df.loc[gain_metrics_df["status"] == "ok"].copy()
if ok_gain_df.empty:
    raise ValueError("No successful gain curves to plot.")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
x = np.arange(len(ok_gain_df))
labels = ok_gain_df["well_name"].tolist()

axes[0].bar(x, ok_gain_df["scale"], color="tab:red", alpha=0.55, label="constant scale")
axes[0].plot(x, ok_gain_df["gain_median"], color="tab:blue", marker="o", lw=1.2, label="median smooth gain")
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=45, ha="right")
axes[0].set_ylabel("Absolute gain")
axes[0].set_title("Constant vs dynamic-gain median")
axes[0].legend(loc="best")
axes[0].grid(True, axis="y", alpha=0.25)

axes[1].bar(x, ok_gain_df["relative_gain_median"], color="tab:green", alpha=0.8)
axes[1].axhline(1.0, color="black", lw=0.9)
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=45, ha="right")
axes[1].set_ylabel("Median relative gain")
axes[1].set_title("gain / constant scale")
axes[1].grid(True, axis="y", alpha=0.25)

width = 0.35
axes[2].bar(x - width / 2, ok_gain_df["constant_nrmse"], width=width, color="tab:red", alpha=0.65, label="constant")
axes[2].bar(
    x + width / 2, ok_gain_df["dynamic_gain_nrmse"], width=width, color="tab:blue", alpha=0.75, label="dynamic gain"
)
axes[2].set_xticks(x)
axes[2].set_xticklabels(labels, rotation=45, ha="right")
axes[2].set_ylabel("NRMSE")
axes[2].set_title("Fit metric")
axes[2].legend(loc="best")
axes[2].grid(True, axis="y", alpha=0.25)

fig.tight_layout()
gain_summary_fig = figure_dir / "qc_01_gain_curve_summary.png"
fig.savefig(gain_summary_fig, dpi=180, bbox_inches="tight")
plt.show()
print("Saved", gain_summary_fig)


## 6) 输出清单


In [ ]:
print("Metrics:")
print(gain_metrics_file)
print("\nFigures:")
for path in sorted(figure_dir.glob("*.png")):
    print(path)
print("\nGain curves:")
for path in sorted(gain_curve_dir.glob("gain_curve_*.csv")):
    print(path)
print("\nDepth-resampled gain curves:")
for path in sorted(gain_curve_dir.glob("gain_depth_curve_*.csv")):
    print(path)
